# Modelo de Probabilidad of Default 

Desarrollo del modelo de Probablidad of Default.

1. Primer enfoque básico: En este primer enfoque concentraremos en construir el modelo sin modificaciones, utilizando las variables del EDA y del VIF, sin contar con los logaritmos. 
2. Enfoque tuneado. 

### Importamos librerias 

In [1]:
import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from lightgbm import LGBMClassifier

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import  RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    log_loss,
    brier_score_loss,
)

from scipy import stats

pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_str_lengths(60)
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titleweight"] = "bold"

SEED = 42
CV_SAMPLE_SIZE = 200_000

### Carga y taxonomia de variables.

Separamos nuestras variables`

In [2]:
Xtr_pl = pl.read_parquet("../data/processed/X_train.parquet")

TARGET = "y"

LEAKAGE_VARS = ['grade']

CAT  = [c for c, dt in Xtr_pl.schema.items() if dt == pl.String]
NUM  = [c for c, dt in Xtr_pl.schema.items() if dt.is_numeric()]

LOG_VARS   = [c for c in NUM if c.startswith("log_")]    # 3 logs creados en 02
FLAG_VARS  = [c for c in NUM if c.startswith("flag_")]   # 5 flags de imputación
NUM_BASE   = [c for c in NUM if c not in LOG_VARS + FLAG_VARS]  # 29 originales
RAW_OF_LOG = [c.removeprefix("log_") for c in LOG_VARS]  # variables que tienen su log
VIF_DROP   = ["monthly_payment"]                         # VIF > 10 (colineal con funded_amnt)

# VARS = NUM + CAT

X_train = Xtr_pl.to_pandas()
X_test  = pl.read_parquet("../data/processed/X_test.parquet").to_pandas()
y_train = pl.read_parquet("../data/processed/y_train.parquet")["y"].to_numpy()
y_test  = pl.read_parquet("../data/processed/y_test.parquet")["y"].to_numpy()

print(f"num_base={len(NUM_BASE)} | logs={len(LOG_VARS)} | flags={len(FLAG_VARS)} | cat={len(CAT)}")

num_base=29 | logs=3 | flags=5 | cat=10


Preparamos las variables categóricas para LightGBM. Esto permite identificar y mantener una codificación consistente entre ambos conjuntos.

Además, evita tratar las categorías como números con un orden artificial. Las categorías no observadas durante el entrenamiento pueden convertirse en valores faltantes (NaN) al aplicar esta transformación.

In [3]:
CATEGORIES = {c: pd.Categorical(X_train[c]).categories for c in CAT}

def to_lgbm(df):
    out = df.copy()
    for c in CAT:
        out[c] = pd.Categorical(out[c], categories=CATEGORIES[c])
    return out

X_train_lgb, X_test_lgb = to_lgbm(X_train), to_lgbm(X_test)

1. Debido a que contamos con mas de ~2M de datos, seleccionaremos una muestra representativa y estratificado de 200_000 prestamos del conjunto de entrenamiento (X_train). La estratificación permite conservar aproximadamente la proporción original de incumplimientos (defaults). 

2. La muestra se dividió en dos subconjuntos, con el fin de evaluar la capacidad de generalizacion con datos cuyas etiquetas no se utilizaron para ajustar los parámetros del modelo. 

    - 80 % (160,000 préstamos): entrenamiento de los modelos.
    - 20 % (40,000 préstamos): validación y comparación del rendimiento.


In [4]:
# generamos los indices de las filas para nuestra muestra en el conjunto de entrenamiento
# muestra de 200_000
idx_dev, _ = train_test_split(
    np.arange(len(y_train)),
    train_size=200_000,
    stratify=y_train,
    random_state=SEED
)

# construimos los datos para CV
X_dev = X_train.iloc[idx_dev]
X_dev_lgb = X_train_lgb.iloc[idx_dev]
y_dev = y_train[idx_dev]

# División: 80 % entrenamiento y 20 % validación
idx_fit, idx_valid = train_test_split(
    np.arange(len(y_dev)),
    test_size=0.20,
    stratify=y_dev,
    random_state=SEED
)

# Conjuntos de entrenamiento
X_fit = X_dev.iloc[idx_fit]
X_fit_lgb = X_dev_lgb.iloc[idx_fit]
y_fit = y_dev[idx_fit]

# Conjuntos de validación
X_valid = X_dev.iloc[idx_valid]
X_valid_lgb = X_dev_lgb.iloc[idx_valid]
y_valid = y_dev[idx_valid]

# configguramos nuestra validación cruzada de 5 folds. 
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

## Construcción de Modelos básicos

### Regresión logística (LR)

Se definieron cuatro variantes para comparar el efecto de las transformaciones logarítmicas y la exclusión de monthly_payment, variable identificada con VIF elevado.

- `C=1.0:` establece una regularización moderada por defecto. En regresión logística, valores menores implican mayor regularización; este parámetro se ajustará posteriormente.
- `solver="lbfgs":` optimizador adecuado para regresión logística multiclase o binaria con regularización L2, especialmente con conjuntos de datos de tamaño considerable.
- `max_iter=1000:` permite suficientes iteraciones para alcanzar la convergencia.
- `class_weight=None:` conserva el peso natural de las clases, evitando reponderar artificialmente los incumplimientos y alterar las probabilidades estimadas.
- `StandardScaler:`estandariza las variables numéricas continuas para facilitar la optimización y evitar que las diferencias de escala dominen el ajuste.
- `OneHotEncoder(handle_unknown="ignore", min_frequency=0.005):` codifica las variables categóricas y agrupa las categorías con frecuencia inferior al 0.5 %, reduciendo la cantidad de columnas y el impacto de niveles poco frecuentes.
- `Variables binarias (FLAG_VARS):` se mantienen sin escalar, ya que representan indicadores de imputación.

In [5]:
def swap_raw_for_log(cols):
    """Sustituye cada variable cruda por su log (si existe)."""
    return [f"log_{c}" if c in RAW_OF_LOG else c for c in cols]

NUM_NOVIF = [c for c in NUM_BASE if c not in VIF_DROP]

CONT_SETS = {
    "LR_A (crudas, todas)" : NUM_BASE,
    "LR_B (crudas, sin VIF)" : NUM_NOVIF,
    "LR_C (logs, todas)": swap_raw_for_log(NUM_BASE),
    "LR_D (logs, sin VIF)": swap_raw_for_log(NUM_NOVIF) 
}

def make_lr(cont_cols):
    """Pipeline LR: el preprocesado se ajusta dentro de cada fold (sin fuga).
    ColumnTransformer descarta (remainder='drop') toda columna no listada, así
    cada variante usa exactamente las variables que declara."""
    prep = ColumnTransformer([
        ("num",  StandardScaler(), cont_cols),          # LR necesita escala común
        ("flag", "passthrough",    FLAG_VARS),          # binarias: sin escalar
        ("cat",  OneHotEncoder(handle_unknown="ignore", # nivel nuevo -> todo ceros
                               min_frequency=0.005,     # niveles raros -> 1 grupo
                               sparse_output=True), CAT),
    ])
    
    model = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        class_weight=None,       # NO reponderar: necesitamos PD calibrada
        random_state=SEED)

    return Pipeline([("prep", prep), ("model", model)])

### LightGBM (LGBM)

Se estableció un modelo base de clasificación binaria con n_estimators=100, es decir, 100 árboles de boosting.

- `objective="binary":` define la tarea como clasificación binaria para estimar la probabilidad de incumplimiento.
- `n_estimators=100:` proporciona un punto de partida sencillo para evaluar el rendimiento antes de ajustar la complejidad del modelo.
- `random_state=SEED:` permite reproducir los resultados bajo las mismas condiciones.
- `n_jobs=-1:` utiliza los núcleos de CPU disponibles para acelerar el entrenamiento.
- `verbosity=-1:`reduce los mensajes de salida durante la ejecución.

LightGBM utiliza las variables categóricas en formato category de pandas, sin necesidad de escalado ni One-Hot Encoding. No se activaron is_unbalance ni scale_pos_weight, para evitar introducir una ponderación artificial de las clases que pueda afectar la interpretación de las probabilidades.


In [6]:
# LGBM sin pipeline: no necesita escalado ni one-hot. Sin logs (invariancia).
FEATURES_LGBM = NUM_BASE + FLAG_VARS + CAT

lgbm_base = LGBMClassifier(
    objective="binary",
    n_estimators=100, 
    # sin is_unbalance / scale_pos_weight, por la misma razón de calibración
    random_state=SEED, 
    n_jobs=-1, 
    verbosity=-1
)

### Métricas de evaluación de los modelos

Se utilizaron cuatro indicadores para evaluar la capacidad predictiva y el comportamiento agregado de las probabilidades de incumplimiento (PD).

- **ROC-AUC:** mide la capacidad del modelo para distinguir entre préstamos que incumplen y los que no, independientemente de un umbral de clasificación. Se prefiere frente a accuracy porque esta última puede resultar engañosa cuando existe desbalance entre clases.

- **LogLoss:** evalúa la calidad de las probabilidades estimadas y penaliza especialmente las predicciones muy seguras que resultan incorrectas. Es fundamental porque el objetivo no es únicamente clasificar préstamos, sino estimar PD para el cálculo de pérdidas crediticias esperadas (ECL).

- **PD media:** representa el promedio de las probabilidades de incumplimiento estimadas. Permite contrastar el nivel de riesgo promedio predicho con la tasa real de incumplimiento de la muestra.

- **SumPD/Defaults:** relaciona la suma de las PD estimadas con el número observado de incumplimientos. Un resultado cercano a 1 indica concordancia agregada entre los defaults esperados y los observados, aunque no garantiza una calibración correcta en todos los segmentos.

¿Por qué no utilizar únicamente métricas clásicas?

Métricas como ``accuracy, precisión, recall y F1-score`` requieren establecer un umbral para convertir probabilidades en clases. Esto puede ser útil para decisiones operativas, pero pierde información sobre la magnitud de las probabilidades.

Para este proyecto,**AUC y LogLoss** permiten evaluar discriminación y calidad probabilística, mientras que PD media y SumPD/Defaults aportan una comprobación agregada relevante para el riesgo crediticio.

Estas métricas evalúan la PD; por sí solas no determinan la calidad final de la ECL, que también depende de la pérdida dado el incumplimiento (LGD) y la exposición al incumplimiento (EAD).

In [7]:
def metricas_pd(y, p, nombre):
    auc = roc_auc_score(y, p)
    ll = log_loss(y, p, labels=[0, 1])

    pd_media = np.mean(p)
    defaults_reales = np.sum(y)

    ratio_pd_defaults = (
        np.sum(p) / defaults_reales
        if defaults_reales > 0
        else np.nan
    )

    return {
        "modelo": nombre,
        "AUC": auc,
        "LogLoss": ll,
        "PD_media": pd_media,
        "SumPD/Defaults": ratio_pd_defaults,
    }


In [8]:
resultados = []

# Modelo para cada grupo de RL
for name, cols in CONT_SETS.items():
    model = make_lr(cols)
    model.fit(X_fit,y_fit)

    #validamos 
    pd_pred = model.predict_proba(X_valid)[:,1]

    resultados.append(metricas_pd(y_valid, pd_pred, name))

#modelo para LGB
lgbm_base.fit(X_fit_lgb[FEATURES_LGBM], y_fit)

#validamos
pd_pred_lgbm = lgbm_base.predict_proba(X_valid_lgb[FEATURES_LGBM])[:,1]

resultados.append(metricas_pd(y_valid, pd_pred_lgbm, "LGBM_base"))

# Tabla comparativa
tabla_modelos = (
    pd.DataFrame(resultados)
    .set_index("modelo")
    .sort_values("LogLoss")
    .round(4)
)

tabla_modelos

,AUC,LogLoss,PD_media,SumPD/Defaults
modelo,,,,
LGBM_base,0.7530,0.3389,0.1312,1.0038
"LR_D (logs, sin VIF)",0.7255,0.3503,0.1316,1.0067
"LR_C (logs, todas)",0.7253,0.3504,0.1316,1.0067
"LR_B (crudas, sin VIF)",0.7251,0.3504,0.1315,1.0059
"LR_A (crudas, todas)",0.7251,0.3504,0.1315,1.0059


- LightGBM obtuvo el mejor rendimiento, con un AUC de 0.7530 y un LogLoss de 0.3389. Esto indica una mayor capacidad para distinguir entre préstamos que incumplen y los que no, así como una mejor calidad de las probabilidades estimadas respecto a las variantes LR evaluadas.

- La regresión logística presentó resultados similares entre sus cuatro variantes. La mejor fue LR_D (logs, sin VIF), aunque su mejora frente a las demás variantes fue mínima. Esto sugiere que las transformaciones logarítmicas y la exclusión de monthly_payment aportaron poca mejora en esta configuración base.

## GrindSearch e hiperparámetros. 

**Qué se hizo.** `GridSearchCV` sobre el pipeline de LR_D (logs, sin VIF) con 9 combinaciones y 5 folds estratificados (45 ajustes) sobre `X_dev` (200k filas):
- **L2** (`lbfgs`): C ∈ {0.001, 0.01, 0.1, 1, 10}
- **L1** (`liblinear`): C ∈ {0.01, 0.1, 1, 10}

**Por qué.**
- C controla cuánto se penaliza la complejidad. Un C pequeño regulariza mucho y puede infraajustar, y uno grande casi no regulariza.
- L1 puede anular coeficientes (selección implícita de variables) y L2 solo los encoge.
- Se optimiza **LogLoss** (`refit="logloss"`) porque la PD debe estar calibrada. El AUC se registra como segunda métrica.
- El preprocesado (escalado, one-hot) va dentro del `Pipeline`, así que se reajusta en cada fold y no hay fuga.


In [ ]:
# Pipeline de LR: preprocesado + modelo en un solo objeto.
# Así GridSearchCV reajusta el escalado/one-hot DENTRO de cada fold (sin fuga).
lr_d = make_lr(CONT_SETS["LR_D (logs, sin VIF)"])

# Lista de diccionarios = rejillas INDEPENDIENTES (no se cruzan entre sí).
# Se usa lista porque lbfgs solo soporta L2, y liblinear soporta L1:
# un diccionario único generaría combinaciones inválidas.
# "model__" es el prefijo: apunta al paso "model" del Pipeline.
param_grid_lr = [
    {
        "model__solver": ["lbfgs"],
        "model__l1_ratio": [0.0],                    # 0.0 = L2 pura (Ridge)
        "model__C": [0.001, 0.01, 0.1, 1, 10],       # 5 combinaciones
    },
    {
        "model__solver": ["liblinear"],
        "model__l1_ratio": [1.0],                    # 1.0 = L1 pura (Lasso)
        "model__C": [0.01, 0.1, 1, 10],              # 4 combinaciones
    },
]                                                    # total: 5 + 4 = 9 candidatos

grid_lr = GridSearchCV(
    estimator=lr_d,
    param_grid=param_grid_lr,
    cv=cv,                                   # 5 folds estratificados, sobre X_dev
    scoring={"logloss": "neg_log_loss", "auc": "roc_auc"},
    refit="logloss",                         # criterio principal: log-loss
    n_jobs=-1,
    verbose=1,
)

# 9 candidatos x 5 folds = 45 ajustes. Al terminar, "refit" reentrena el
# mejor candidato con TODO X_dev y lo deja en grid_lr.best_estimator_.
grid_lr.fit(X_dev, y_dev)

# cv_results_ guarda un resumen por candidato; se arma una tabla legible.
res_lr = (
    pd.DataFrame(grid_lr.cv_results_)
    .assign(
        LogLoss=lambda d: -d["mean_test_logloss"],   # se invierte el signo para leerlo normal
        LogLoss_std=lambda d: d["std_test_logloss"], # variabilidad entre folds
    )
    [[
        "param_model__solver", "param_model__l1_ratio", "param_model__C",
        "LogLoss", "LogLoss_std", "mean_test_auc", "std_test_auc",
    ]]
    .sort_values("LogLoss")                          # mejor (menor LogLoss) arriba
    .round(4)
)

print(res_lr.to_string(index=False))
print("Mejores parámetros LR:", grid_lr.best_params_)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
param_model__solver  param_model__l1_ratio  param_model__C  LogLoss  LogLoss_std  mean_test_auc  std_test_auc
          liblinear                    1.0           1.000   0.3505       0.0010         0.7252        0.0034
          liblinear                    1.0          10.000   0.3505       0.0010         0.7252        0.0034
          liblinear                    1.0           0.100   0.3505       0.0010         0.7252        0.0035
              lbfgs                    0.0           0.100   0.3505       0.0010         0.7251        0.0034
              lbfgs                    0.0           1.000   0.3505       0.0010         0.7251        0.0034
              lbfgs                    0.0          10.000   0.3505       0.0010         0.7250        0.0034
              lbfgs                    0.0           0.010   0.3506       0.0011         0.7252        0.0035
              lbfgs                    0.0           0.001  

**Qué se hizo.** `RandomizedSearchCV` con 20 combinaciones sorteadas del espacio de hiperparámetros, 5 folds estratificados (100 ajustes) sobre `X_dev` (200k filas, 44 variables sin los `log_`). Criterio: **LogLoss**.

**Por qué.**
- Un GBM tiene mucha capacidad y puede memorizar el train. Los hiperparámetros controlan cuánto puede crecer cada árbol y cuánto se regulariza.
- Se usa búsqueda aleatoria y no rejilla porque con 8 hiperparámetros una rejilla es inviable. La búsqueda aleatoria cubre mejor los rangos importantes con pocas pruebas.
- Se optimiza LogLoss y no AUC porque la PD tiene que estar calibrada.
- Se mantiene `n_estimators=100` fijo y sin reponderar clases (`is_unbalance` / `scale_pos_weight`), por calibración.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, loguniform, uniform

# LightGBM no necesita escalado ni one-hot: solo las features propias del GBM
# (sin los log_*, porque los árboles son invariantes a transformaciones monótonas).
XD = X_dev_lgb[FEATURES_LGBM].copy()
yD = np.asarray(y_dev).ravel()          # aplana a vector 1D

# Espacio de búsqueda. Dos tipos de entradas:
#  - distribuciones de scipy (se SORTEA un valor continuo/entero)
#  - listas (se sortea uno de los valores)
param_space = {
    "learning_rate": loguniform(0.02, 0.12),  # log-uniforme: explora por órdenes de magnitud
    "num_leaves": randint(7, 64),             # entero en [7, 63] (el límite superior no entra)
    "max_depth": [-1, 4, 6, 8, 10],           # -1 = sin límite de profundidad
    "min_child_samples": [50, 100, 200, 400, 800],  # mínimo de filas por hoja: más alto = más suave
    "subsample": uniform(0.6, 0.4),           # uniform(loc, scale) = [0.6, 1.0], NO [0.6, 0.4]
    "colsample_bytree": uniform(0.6, 0.4),    # fracción de columnas por árbol: [0.6, 1.0]
    "reg_alpha": loguniform(1e-3, 5),         # regularización L1 sobre pesos de hojas
    "reg_lambda": loguniform(0.1, 20),        # regularización L2 sobre pesos de hojas
}

search_lgbm = RandomizedSearchCV(
    estimator=lgbm_base,            # trae n_estimators=100 y random_state fijos
    param_distributions=param_space,
    n_iter=20,                      # 20 combinaciones sorteadas
    scoring="neg_log_loss",         # sklearn maximiza: -LogLoss más alto = mejor
    cv=cv,                          # mismos 5 folds que la LR: comparación justa
    refit=True,                     # reentrena el mejor con todo XD -> best_estimator_
    random_state=SEED,              # hace reproducible el SORTEO de combinaciones
    n_jobs=1,                       # LightGBM ya paraleliza internamente (n_jobs=-1)
    verbose=2,
    return_train_score=True,        # guarda score en train para ver sobreajuste por candidato
    error_score="raise",            # si algo falla, que pare (no lo oculte con NaN)
)

search_lgbm.fit(XD, yD)             # 20 x 5 = 100 ajustes

print("Mejores hiperparámetros:")
print(search_lgbm.best_params_)
print(f"\nMejor LogLoss CV: {-search_lgbm.best_score_:.6f}")

# Tabla completa de candidatos ordenada del mejor al peor.
resultados_lgbm = (
    pd.DataFrame(search_lgbm.cv_results_)
    .sort_values("rank_test_score")
)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV] END colsample_bytree=0.749816047538945, learning_rate=0.10985745201142037, max_depth=6, min_child_samples=800, num_leaves=27, reg_alpha=0.003776663327107336, reg_lambda=0.22853255256339208, subsample=0.6232334448672797; total time=   3.6s
[CV] END colsample_bytree=0.749816047538945, learning_rate=0.10985745201142037, max_depth=6, min_child_samples=800, num_leaves=27, reg_alpha=0.003776663327107336, reg_lambda=0.22853255256339208, subsample=0.6232334448672797; total time=   2.1s
[CV] END colsample_bytree=0.749816047538945, learning_rate=0.10985745201142037, max_depth=6, min_child_samples=800, num_leaves=27, reg_alpha=0.003776663327107336, reg_lambda=0.22853255256339208, subsample=0.6232334448672797; total time=   1.9s
[CV] END colsample_bytree=0.749816047538945, learning_rate=0.10985745201142037, max_depth=6, min_child_samples=800, num_leaves=27, reg_alpha=0.003776663327107336, reg_lambda=0.22853255256339208, subsample=0

**Qué se hizo.** Con los mejores modelos de cada búsqueda se generaron predicciones **out-of-fold (OOF)** sobre `X_dev` (200k filas) con los mismos 5 folds estratificados. Cada préstamo recibe su PD de un modelo que **no lo vio al entrenar**, y con esas 200k predicciones se calculan las métricas globales.

**Por qué.**
- Es la comparación más justa entre modelos: ambos se evalúan con las mismas filas y los mismos folds, y todas las filas actúan como validación.
- Las PD OOF son además la materia prima de las fases siguientes (calibración por tramos y margen conservador del ECL), porque son PD "honestas" sobre datos no vistos.
- Se clona el estimador (`clone`) para partir de un modelo sin entrenar, con los hiperparámetros ganadores.

In [ ]:
from sklearn.model_selection import cross_val_predict

yD = np.asarray(y_dev).ravel()

# clone() crea una copia SIN entrenar, con los mismos hiperparámetros ganadores.
# Así cross_val_predict entrena desde cero en cada fold (no reutiliza lo ya ajustado).
lr_best = clone(grid_lr.best_estimator_)

# Para cada fold: entrena con 4/5 de los datos y predice el 1/5 restante.
# Al juntar los 5 trozos, cada fila tiene una predicción de un modelo que NO la vio.
# method="predict_proba" devuelve [P(y=0), P(y=1)]; [:, 1] es la PD.
p_lr_oof = cross_val_predict(
    lr_best, 
    X_dev, 
    yD,
    cv=cv, 
    method="predict_proba", 
    n_jobs=1
)[:, 1]

# Mismo procedimiento con LightGBM, pero con SUS features (XD, sin los log_*).
lgbm_best = clone(search_lgbm.best_estimator_)
p_lgbm_oof = cross_val_predict(
    lgbm_best, XD, yD, cv=cv, method="predict_proba", n_jobs=1
)[:, 1]

# Tabla comparativa con tu función de métricas (AUC, LogLoss, PD media, SumPD/Defaults)
comparacion_dev = pd.DataFrame([
    metricas_pd(yD, p_lr_oof, "LR tuneada"),
    metricas_pd(yD, p_lgbm_oof, "LightGBM tuneado"),
])

display(
    comparacion_dev.style.format({
        "LogLoss": "{:.4f}",
        "AUC": "{:.4f}",
        "PD_media": "{:.2%}",
        "SumPD/Defaults": "{:.4f}",   # 4 decimales: con 2 todo parece "1.00"
    })
)

,modelo,AUC,LogLoss,PD_media,SumPD/Defaults
0,LR tuneada,0.7252,0.3505,13.08%,1.00
1,LightGBM tuneado,0.7541,0.3386,13.06%,1.00


**Lectura.**
- **Discriminación:** LightGBM separa mejor buenos y malos pagadores (+0.029 de AUC).
- **Calidad probabilística:** LightGBM tiene un LogLoss menor en 0.012, una diferencia mucho mayor que la desviación entre folds (±0.001), así que es una mejora real.
- **Calibración global:** ambos modelos prevén el total de impagos casi exactamente (ratio ≈ 1.00; PD media ≈ tasa de impago real, ~13.1%).
- Los valores coinciden con los de la validación cruzada (0.3505 y 0.3386), lo que confirma que el procedimiento es consistente.
- **Pendiente:** que el total cuadre no implica que cada tramo de riesgo esté bien calibrado. Se revisará por deciles.

### Diagnóstico de sobreajuste: gap train vs. validación por fold

**Qué se hizo.** Para cada modelo (LR y LightGBM con sus hiperparámetros ganadores) y para cada uno de los 5 folds se reentrenó el modelo con 4/5 de `X_dev`. Se midieron AUC y LogLoss tanto en esas filas de entrenamiento como en el 1/5 de validación, y se calculó el **gap** (diferencia entre ambos).

**Por qué.**
- Un buen resultado en validación no basta. Hay que saber *cuánto* del rendimiento del modelo en train es memorización, porque eso dice si es estable y si lo que vemos se sostendrá en datos nuevos.
- Se mira por fold (no solo la media) para ver si el gap es sistemático o depende de la partición.
- Convención del gap: `AUC_train − AUC_val` y `LogLoss_val − LogLoss_train`. En ambos casos **positivo = sobreajuste**.

In [ ]:

from sklearn.base import clone
from sklearn.metrics import roc_auc_score, log_loss

# Cada modelo con SU matriz de datos: la LR usa X_dev (su Pipeline hace el preprocesado),
# LightGBM usa XD (solo sus features, categóricas en dtype 'category').
modelos = {
    "LR tuneada": (grid_lr.best_estimator_, X_dev),
    "LightGBM tuneado": (search_lgbm.best_estimator_, XD),
}

resultados_gap = []

for nombre, (modelo, X) in modelos.items():
    # cv.split devuelve POSICIONES (índices enteros) de train y validación.
    # Con random_state fijo, genera los mismos folds que en el tuning y en OOF.
    for fold, (idx_train, idx_val) in enumerate(cv.split(X, yD), start=1):

        # clone: copia sin entrenar. Cada fold parte de cero (sin arrastrar el ajuste anterior).
        modelo_fold = clone(modelo)
        modelo_fold.fit(X.iloc[idx_train], yD[idx_train])    # solo 4/5 de los datos

        # Predice sobre lo que VIO (train) y sobre lo que NO vio (validación).
        p_train = modelo_fold.predict_proba(X.iloc[idx_train])[:, 1]
        p_val = modelo_fold.predict_proba(X.iloc[idx_val])[:, 1]

        # Evita log(0) si alguna probabilidad es exactamente 0 o 1.
        p_train_segura = np.clip(p_train, 1e-15, 1 - 1e-15)
        p_val_segura = np.clip(p_val, 1e-15, 1 - 1e-15)

        auc_train = roc_auc_score(yD[idx_train], p_train)
        ll_train = log_loss(yD[idx_train], p_train_segura, labels=[0, 1])
        auc_val = roc_auc_score(yD[idx_val], p_val)
        ll_val = log_loss(yD[idx_val], p_val_segura, labels=[0, 1])

        resultados_gap.append({
            "Modelo": nombre, "Fold": fold,
            "AUC_train": auc_train, "AUC_val": auc_val,
            "Gap_AUC": auc_train - auc_val,         # AUC: más alto es mejor -> train - val
            "LogLoss_train": ll_train, "LogLoss_val": ll_val,
            "Gap_LogLoss": ll_val - ll_train,       # LogLoss: más bajo es mejor -> val - train
        })

df_gap = pd.DataFrame(resultados_gap)
# ... (tablas por fold y resumen con media y std, como ya lo tienes)

Modelo,Fold,AUC_train,AUC_val,Gap_AUC,LogLoss_train,LogLoss_val,Gap_LogLoss
LR tuneada,1,0.72764,0.72490,0.00275,0.34962,0.35057,0.00095
LR tuneada,2,0.72743,0.72504,0.00239,0.34962,0.35077,0.00115
LR tuneada,3,0.72837,0.72113,0.00724,0.34944,0.35140,0.00196
LR tuneada,4,0.72766,0.72367,0.00399,0.34951,0.35112,0.00161
LR tuneada,5,0.72581,0.73142,-0.00561,0.35016,0.34851,-0.00166
LightGBM tuneado,1,0.78911,0.75708,0.03203,0.32346,0.33740,0.01394
LightGBM tuneado,2,0.79151,0.75221,0.03931,0.32238,0.33960,0.01722
LightGBM tuneado,3,0.79037,0.75033,0.04004,0.32294,0.34002,0.01707
LightGBM tuneado,4,0.79149,0.75121,0.04028,0.32235,0.33997,0.01762
LightGBM tuneado,5,0.78870,0.76010,0.02860,0.32379,0.33579,0.01200


AUC_train           AUC_val           Gap_AUC           \
                      mean      std     mean      std     mean      std   
Modelo                                                                    
LR tuneada         0.72738  0.00095  0.72523  0.00380  0.00215  0.00474   
LightGBM tuneado   0.79024  0.00131  0.75419  0.00421  0.03605  0.00539   

                 LogLoss_train          LogLoss_val          Gap_LogLoss  \
                          mean      std        mean      std        mean   
Modelo                                                                     
LR tuneada             0.34967  0.00029     0.35047  0.00114     0.00080   
LightGBM tuneado       0.32299  0.00064     0.33856  0.00189     0.01557   

                           
                      std  
Modelo                     
LR tuneada        0.00143  
LightGBM tuneado  0.00248

**Lectura.**
- **LR: sin sobreajuste.** El gap medio (0.002 AUC, 0.0008 LogLoss) es menor que su propia desviación entre folds, es decir, **indistinguible de cero**. En el fold 5 incluso es negativo (la validación salió mejor que el train). Lo que limita a la LR es su capacidad, no la memorización.
- **LightGBM: sobreajuste moderado y estable.** El gap es positivo en los 5 folds (AUC 0.029–0.040) y con poca dispersión, así que es una propiedad del modelo y no un accidente de una partición.
- **Lo que importa es la validación, y ahí LightGBM gana en cada fold:** supera a la LR entre +0.027 y +0.032 de AUC, una diferencia mucho mayor que la variabilidad entre folds (±0.004).
- El gap de LightGBM es el precio de su mayor capacidad: el tuning ya buscó el mejor LogLoss en validación, así que reducir el gap a la fuerza (más regularización) no garantiza mejorar la validación.
- **Conclusión:** el sobreajuste de LightGBM es aceptable y no cambia la decisión. Se mantiene LightGBM como modelo de PD, con el test como confirmación final.

## Evaluación final en el test reservado

**Qué se hizo.** Se reentrenaron las dos configuraciones ganadoras (LR con L1 C=1 y LightGBM con los hiperparámetros del random search) sobre **toda la muestra de desarrollo** (`X_dev`, 200k filas) y se predijo una sola vez sobre el **test reservado**, que no se usó en ningún paso de limpieza, ajuste ni selección.

**Por qué.**
- El CV y el OOF sirvieron para *elegir*. Solo el test, que ningún paso de selección ha visto, permite *confirmar* que lo elegido generaliza.
- Se reutilizan las mismas métricas (`metricas_pd`) que en desarrollo para que los números sean comparables.
- Cada modelo recibe sus propias variables: la LR usa `X_test` (su Pipeline hace el preprocesado) y LightGBM usa `X_test_lgb[FEATURES_LGBM]` (mismas 44 columnas del entrenamiento).

In [ ]:
python
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, log_loss

# Modelos finales: copias SIN entrenar de los ganadores, ajustadas con TODO X_dev.
# (Ya no hay folds: el CV sirvió para elegir; ahora se entrena una vez con todo lo disponible.)
lr_final = clone(grid_lr.best_estimator_)
lr_final.fit(X_dev, yD)

lgbm_final = clone(search_lgbm.best_estimator_)
lgbm_final.fit(XD, yD)

y_test_array = np.asarray(y_test).ravel()

# La LR recibe el DataFrame completo: su ColumnTransformer selecciona y transforma columnas.
p_lr_test = lr_final.predict_proba(X_test)[:, 1]

# LightGBM necesita EXACTAMENTE las columnas del entrenamiento (44), en el mismo orden
# y con categóricas en dtype 'category' con las categorías de train (to_lgbm).
p_lgbm_test = lgbm_final.predict_proba(X_test_lgb[FEATURES_LGBM])[:, 1]

comparacion_test = pd.DataFrame([
    metricas_pd(y_test_array, p_lr_test, "LR final"),
    metricas_pd(y_test_array, p_lgbm_test, "LightGBM final"),
])
display(comparacion_test.style.format({
    "AUC": "{:.5f}", "LogLoss": "{:.5f}",
    "PD_media": "{:.4%}", "SumPD/Defaults": "{:.4f}",
}).hide(axis="index"))


modelo,AUC,LogLoss,PD_media,SumPD/Defaults
LR final,0.72565,0.35024,13.0981%,1.0019
LightGBM final,0.75548,0.33788,13.0814%,1.0006


**Lectura.**
- **Generalización confirmada.** Las métricas en test son prácticamente iguales a las de desarrollo (ligeramente mejores). Las diferencias son del orden del error de muestreo, así que no indican "mejora real", solo que no hay deterioro. No hay señales de sobreajuste al CV ni de desajuste entre desarrollo y test.
- **La ventaja de LightGBM se mantiene:** +0.030 de AUC y −0.012 de LogLoss sobre la LR, igual que en validación.
- **Calibración global muy buena.** La tasa real de impago del test es ≈ 13.07%. LightGBM predice 13.081% (ratio 1.0006, +0.06% sobre los impagos reales) y la LR 13.098% (+0.19%). La PD total cuadra con la realidad.
- **Decisión: PD = LightGBM.**
- **Limitaciones.** Ambos modelos se entrenaron con 200k filas y no con todo el train; sin early stopping (n_estimators=100 fijo); la calibración por tramos de riesgo no se ha comprobado todavía. Que el total cuadre no garantiza que cada decil lo haga.
- **Disciplina.** El test ya se ha mirado: no se re-tunea a partir de él. Cualquier mejora (más datos, más árboles) se decide con CV/OOF y el test solo se vuelve a consultar al final.

In [ ]:
PRED_DIR = Path("../data/processed")
PRED_DIR.mkdir(parents=True, exist_ok=True)

# Train: PD out-of-fold (cada fila predicha por un modelo que NO la vio)
pd.DataFrame({
    "pd_lgbm": p_lgbm_oof,       # PD del modelo ganador
    "pd_lr": p_lr_oof,           # opcional: útil como referencia
    "y": yD,
}, index=X_dev.index).to_parquet(PRED_DIR / "pd_train_oof.parquet")

# Test: PD del modelo final
pd.DataFrame({
    "pd_lgbm": p_lgbm_test,
    "pd_lr": p_lr_test,
    "y": y_test_array,
}, index=X_test.index).to_parquet(PRED_DIR / "pd_test.parquet")

## Reentrenamos con todos los valores el modelo elegido

**Por qué se reentrena y no se "extrapola".** El modelo de 200k se descarta. Solo se reutilizan sus hiperparámetros: el modelo se entrena de nuevo con todos los datos, porque un GBM mejora con más ejemplos (más hojas con soporte estadístico suficiente).

**Ajuste por tamaño de datos.** `min_child_samples` cuenta filas, así que se escaló en proporción al tamaño del train (×8,56: de 800 a ≈6.850) para mantener la misma suavidad que en el tuning. El resto de hiperparámetros no depende del tamaño.

**Qué se generó y por qué.**
- **PD out-of-fold sobre todo el train** (5 folds, misma `SEED`). Cada préstamo recibe la PD de un modelo que no lo vio al entrenar. Es la que se usará para calcular el ECL de train y calibrar el margen conservador, porque simula cómo se comportará el modelo con préstamos nuevos. Predecir el train con el modelo final daría PD demasiado optimistas (el modelo ya conoce esas filas) y un margen insuficiente.
- **PD de test** con el modelo final, entrenado con todo el train. Es la PD que se entrega.

**Resultados.**


In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# 1) Datos: todo el train, solo con las features de LightGBM
X_full = X_train_lgb[FEATURES_LGBM]
y_full = np.asarray(y_train).ravel()

# 2) Hiperparámetros ganadores de la búsqueda (hecha con 200k filas).
#    min_child_samples es un CONTEO de filas: se escala con el tamaño de datos.
params = search_lgbm.best_params_.copy()
escala = len(X_full) / len(XD)                       # 1.711.714 / 200.000 = 8,56
params["min_child_samples"] = int(params["min_child_samples"] * escala)

# clone = modelo sin entrenar, con los parámetros ya escalados.
lgbm_full = clone(search_lgbm.best_estimator_).set_params(**params)

# 3) OOF sobre TODO el train: cada fila se predice con un modelo que no la vio.
cv_full = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
p_lgbm_oof_full = cross_val_predict(
    lgbm_full, X_full, y_full, cv=cv_full, method="predict_proba", n_jobs=1
)[:, 1]

# 4) Modelo final: se entrena con todo el train y predice el test.
lgbm_final_full = clone(lgbm_full).fit(X_full, y_full)
p_lgbm_test_full = lgbm_final_full.predict_proba(X_test_lgb[FEATURES_LGBM])[:, 1]



{'modelo': 'LGBM OOF train completo', 'AUC': 0.7570746345575077, 'LogLoss': 0.3371675489791479, 'PD_media': np.float64(0.1307262726925944), 'SumPD/Defaults': np.float64(0.9999686786865776)}
{'modelo': 'LGBM final test', 'AUC': 0.7592863091135904, 'LogLoss': 0.33623902039943, 'PD_media': np.float64(0.13071407771666546), 'SumPD/Defaults': np.float64(0.9998810318219427)}


In [27]:
from pathlib import Path

PRED_DIR = Path("../data/processed")
PRED_DIR.mkdir(parents=True, exist_ok=True)

# 6) Guardar (el índice permite unir con LGD y EAD)
pd.DataFrame({"pd_lgbm": p_lgbm_oof_full, "y": y_full}, index=X_train.index) \
  .to_parquet(PRED_DIR / "pd_train_oof.parquet")
pd.DataFrame({"pd_lgbm": p_lgbm_test_full, "y": np.asarray(y_test).ravel()}, index=X_test.index) \
  .to_parquet(PRED_DIR / "pd_test.parquet")